# 4.6 隱藏向量怎麼變成下一字的分數？

# 第 4 章：第一個 Dense Transformer 語言模型

前置：第3章的因果 attention。完整可讀實作在 tiny_perceptron/model.py；預設只用 learned position、LayerNorm、Dense FFN，還沒有 FlashAttention 或 MoE。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：像原稿旁加修訂，而不是撕掉原稿**

y=x+f(x)，原資料和修訂都有路徑。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/residual.svg")))

**先想一想：**hidden size=8，詞表=20，最後一軸是幾？

每個隱藏向量還不是字。輸出Linear為詞表中每個候選給一個分數，再用cross-entropy比較正確答案。

**把直覺寫成數學：**$[B,T,D]\times[D,V]\to[B,T,V]$；V與tokenizer詞表一致。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.model import TinyLM, ModelConfig

model = TinyLM(ModelConfig(vocab_size=20, width=8))
logits = model(torch.tensor([[1, 2, 3]]))["logits"]
print(logits.shape)
assert logits.shape == (1, 3, 20)

**如何讀結果：**logits可以是任意實數，還不是機率；ID最大值不得超過詞表。

**只改一件事：**只改vocab_size，查看參數量與最後一軸。
